In [1]:
! nvidia-smi

Wed May  6 12:50:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
! nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [14]:
%%writefile ./a4.cu
#include <iostream>
using namespace std;

__global__ void add(int *A, int *B, int *C) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    C[i] = A[i] + B[i];
}

int main() {
    int N = 4;
    int A[] = {1,2,3,4};
    int B[] = {5,6,7,8};
    int C[4];

    int *dA, *dB, *dC;

    cudaMalloc(&dA, N*sizeof(int));
    cudaMalloc(&dB, N*sizeof(int));
    cudaMalloc(&dC, N*sizeof(int));

    cudaMemcpy(dA, A, N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, B, N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(dC, C, N*sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    add<<<blocks, threadsPerBlock>>>(dA, dB, dC);

    cudaMemcpy(C, dC, N*sizeof(int), cudaMemcpyDeviceToHost);

    for(int i=0;i<N;i++){
        cout << C[i] << " ";
    }

    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
}

Writing ./a4.cu


In [15]:
! nvcc a4.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [16]:
! ./a.out

6 8 10 12 